# Extraction de polygones de bâtiments

Suite au tutoriel `Geo_Inference`, nous souhaitons maintenant convertir les masques extraits irréguliers en polygones de bâtiments régularisés.

## Préparation de l'environnement Python

Pour commencer, nous devons installer le package geo-inference dans notre environnement. Nous supposerons que nous avons déjà créé un environnement "tutorial" en utilisant les fichiers de requirements disponibles à la racine de ce dépôt.
- [Si vous avez un GPU disponible sur votre appareil](https://github.com/geoaiclassroom/geoai_learning/blob/main/requirements_gpu.txt)
- [Si vous n'avez que le CPU disponible sur votre appareil](https://github.com/geoaiclassroom/geoai_learning/blob/main/requirements_cpu.txt)

Ensuite, nous devons ajouter geo-inference à cet environnement en utilisant les mêmes instructions que le tutoriel `Geo_Inference`.

Ensuite, activez votre environnement "tutorial" (*conda activate tutorial*) et installez le package `buildingregulariser`.

Ensuite, installez le package :

    pip install buildingregulariser==0.2.4 
    pip install --force-reinstall numpy==1.26.4


In [ ]:
# Exécutez cette cellule pour corriger l'erreur "RuntimeError: This event loop is already running" lors de l'exécution de l'inférence dans le notebook. Il s'agit d'un problème connu avec les notebooks Jupyter et le code asynchrone, et nest_asyncio est une solution de contournement courante.
# De plus, en raison de certains conflits avec nos versions rasterio/gdal, l'ordre d'importation de torch par rapport à gdal est important - nous devons donc importer torch en premier pour éviter les conflits de bibliothèques dynamiques.
import torch
import nest_asyncio
nest_asyncio.apply()

In [ ]:
# Exécutez cette cellule comme monkeypatch si vous exécutez ce tutoriel sur une machine Windows.

# Solution de contournement pour le verrouillage de fichiers Windows lors d'écritures chunked rioxarray+dask.
# geo_inference passe un verrou, ce qui fait que les écritures rioxarray rouvrent le même TIFF de sortie en r+ par chunk ;
# cette opération peut échouer sur Windows avec une erreur exprimant "file used by other process".

import rioxarray.raster_array as rxr_raster_array

_original_to_raster = rxr_raster_array.RasterArray.to_raster

def _to_raster_windows_safe(self, *args, **kwargs):
    kwargs["lock"] = False
    kwargs["windowed"] = False
    return _original_to_raster(self, *args, **kwargs)

rxr_raster_array.RasterArray.to_raster = _to_raster_windows_safe
print("Applied Windows-safe rioxarray to_raster patch (lock=False, windowed=False).")

Applied Windows-safe rioxarray to_raster patch (lock=False, windowed=False).


In [3]:
import zipfile
from pathlib import Path
from datetime import datetime
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import rasterio
import geopandas as gpd
import os
from buildingregulariser import regularize_geodataframe
from shapely.geometry import Polygon, MultiPolygon
from geo_inference.geo_inference import GeoInference

In [ ]:
zip_path = Path("./Data/waterloo_crop_2020.zip")
extract_dir = zip_path.with_suffix("")  # supprime .zip

# Décompresser uniquement si ce n'est pas déjà fait
if not extract_dir.exists():
    print(f"Extracting to {extract_dir}...")
    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(extract_dir)
else:
    print(f"{extract_dir} already extracted.")

Data\waterloo_crop_2020 already extracted.


## Exécution de l'inférence
Nous pourrions soit exécuter l'inférence sur notre fichier geotif en utilisant des arguments de ligne de commande, soit en définissant tout dans un fichier de configuration. Le dépôt geo-inference vous fournit des détails sur la façon dont un fichier de configuration pourrait être défini. Pour plus de clarté, dans ce tutoriel, nous définirons tout comme arguments.

In [ ]:
# Répertoire de sortie unique par exécution pour éviter les collisions de chemins
base_work_dir = Path(r"./Data/waterloo_crop_2020")
run_work_dir = base_work_dir / f"run_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
run_work_dir.mkdir(parents=True, exist_ok=True)

# Initialiser l'objet GeoInference
geo_inf = GeoInference(
    model=r"./logs/mlrun/scripted.pt",
    work_dir=str(run_work_dir),
    mask_to_vec=True,
    mask_to_yolo=False,
    mask_to_coco=False, 
    device="gpu",
    multi_gpu=False,
    gpu_id=0, 
    num_classes=2,
    prediction_threshold=0.5,
    transformers=True,
    transformer_flip=False,
    transformer_rotate=True,
)

In [ ]:
# Effectuer l'extraction de caractéristiques sur une image TIFF
image_path = r"./Data/waterloo_crop_2020/sample.tif"
bands_requested = [1, 2, 3] # bandes RGB
workers = 1
patch_size = 512
# bbox = "0, 0, 1000, 1000"
geo_inf(
    inference_input = image_path,  
    bands_requested = bands_requested, 
    patch_size = patch_size, 
    workers = workers, 
    bbox=None
)

## Régularisation des bâtiments

- D'abord, nous convertissons l'entrée geojson en geodataframe
- Ensuite, nous effectuons un nettoyage de base ; par ex. suppression des petits artefacts et suppression des trous à l'intérieur des polygones
- Enfin, nous appliquons le régularisateur sur le geodataframe et stockons les résultats sous forme de geopackage

In [7]:
candidates = sorted(Path(run_work_dir).glob("*_polygons_*.geojson"), key=lambda p: p.stat().st_mtime)
if not candidates:
    raise FileNotFoundError(f"No GeoJson file found in {run_work_dir}. Run inference first.")
geojson_path = candidates[-1]

In [ ]:
def remove_small_holes(geom, min_area):
    if geom is None:
        return None

    if geom.geom_type == "Polygon":
        # conserver uniquement les trous plus grands que le seuil
        new_holes = []
        for hole in geom.interiors:
            hole_poly = Polygon(hole)
            if hole_poly.area >= min_area:
                new_holes.append(hole)

        return Polygon(geom.exterior, new_holes)

    elif geom.geom_type == "MultiPolygon":
        return MultiPolygon([remove_small_holes(p, min_area) for p in geom.geoms])

    else:
        return geom

def vectorize_instances_to_gpkg(input_geojson, output_path, min_building_area):
    
    gdf = gpd.read_file(input_geojson)
    gdf = gdf[gdf.geometry.area >= min_building_area]
    
    gdf["geometry"] = gdf["geometry"].apply(
        lambda g: remove_small_holes(g, min_building_area)
    )

    regularized = regularize_geodataframe(
        gdf,
        parallel_threshold=3,  # Des valeurs plus élevées permettent moins d'alignement des arêtes
        simplify_tolerance=3,  # Contrôle le niveau de simplification, devrait être 2-3 x la taille de pixel raster
        allow_45_degree=True,  # Activer les angles de 45 degrés
        allow_circles=False,  # Activer la détection de cercles
        circle_threshold=0.98,  # Seuil IOU pour la détection de cercles
        neighbor_alignment = False,  # Après régularisation, essayer d'aligner chaque bâtiment avec les bâtiments voisins
        neighbor_search_distance= 100.0,  # La distance de recherche autour de chaque bâtiment pour trouver des voisins
        neighbor_max_rotation=10  # La rotation maximale autorisée pour s'aligner avec les voisins
        )

    regularized.to_file(output_path, driver="GPKG")

In [9]:
vectorize_instances_to_gpkg(geojson_path, os.path.join(run_work_dir, "regularized_buildings.gpkg"), min_building_area=200)

[2026-06-04 15:38:43,542][pyogrio._io][INFO] - Created 6 records
